<a href="https://colab.research.google.com/github/ANMOLGOLA/ML_Manthan/blob/main/Loan_Eligibility_ML_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Libraries Install & Import
!pip install pandas numpy xgboost scikit-learn pdfplumber

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# 2. Dataset Load & Clean
df = pd.read_csv('loan_approval_dataset.csv')
df.columns = df.columns.str.strip()

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

# 3. Feature Engineering (Income & Target Calculation)
df['monthly_income'] = df['income_annum'] / 12.0
total_assets = df['residential_assets_value'] + df['commercial_assets_value'] + df['luxury_assets_value'] + df['bank_asset_value']
df['estimated_monthly_emi'] = total_assets * 0.005

cibil_multiplier = (df['cibil_score'] / 900) ** 2
disposable_monthly = np.maximum(df['monthly_income'] - df['estimated_monthly_emi'], df['monthly_income'] * 0.3)
df['target_max_loan'] = (disposable_monthly * 0.5 * (df['loan_term'] * 12)) * cibil_multiplier

# 4. Train Model
X = df[['no_of_dependents', 'income_annum', 'loan_term', 'cibil_score',
        'residential_assets_value', 'commercial_assets_value',
        'luxury_assets_value', 'bank_asset_value', 'monthly_income']]
y = df['target_max_loan']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = xgb.XGBRegressor(n_estimators=150, learning_rate=0.05, max_depth=6, random_state=42)
model.fit(X_train, y_train)

# 5. Output Result
y_pred = model.predict(X_test)
print(f"\nSUCCESS! Model Training Complete.")
print(f"Model Accuracy (R2 Score): {r2_score(y_test, y_pred):.2f}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 57.5 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0

SUCCESS! Model Training Complete.
Model Accuracy (R2 Score): 0.99


In [2]:
# Custom Customer Prediction Function
def predict_max_loan(dependents, annual_income, loan_term_years, cibil_score,
                     res_assets, comm_assets, lux_assets, bank_assets):

    monthly_inc = annual_income / 12.0

    input_data = pd.DataFrame([{
        'no_of_dependents': dependents,
        'income_annum': annual_income,
        'loan_term': loan_term_years,
        'cibil_score': cibil_score,
        'residential_assets_value': res_assets,
        'commercial_assets_value': comm_assets,
        'luxury_assets_value': lux_assets,
        'bank_asset_value': bank_assets,
        'monthly_income': monthly_inc
    }])

    # Model se max loan amount calculate karwao
    max_amount = model.predict(input_data)[0]
    return max_amount

# Example Customer Details:
# Annual Income: 12,000,000 | CIBIL: 750 | Tenure: 10 years
predicted_amount = predict_max_loan(
    dependents=2,
    annual_income=12000000,
    loan_term_years=10,
    cibil_score=750,
    res_assets=20000000,
    comm_assets=5000000,
    lux_assets=3000000,
    bank_assets=4000000
)

print("---------------------------------------------")
print(f"Predicted Max Eligible Loan Amount: ₹{predicted_amount:,.2f}")
print("---------------------------------------------")


---------------------------------------------
Predicted Max Eligible Loan Amount: ₹21,545,616.00
---------------------------------------------


In [3]:
import pdfplumber
import re

# ==========================================
# STEP 1: PDF OCR & EXTRACTION FUNCTION
# ==========================================
def extract_financials_from_pdf(pdf_path):
    extracted_data = {
        'monthly_income': 0.0,
        'bank_assets': 0.0
    }

    try:
        with pdfplumber.open(pdf_path) as pdf:
            full_text = ""
            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    full_text += text + "\n"

            # Regular Expression patterns to scan PDF text
            income_match = re.search(r'(?:Net Pay|Total Salary|Monthly Income|Salary):\s*₹?\s*([\d,]+(?:\.\d{2})?)', full_text, re.IGNORECASE)
            bank_match = re.search(r'(?:Balance|Bank Balance|Savings):\s*₹?\s*([\d,]+(?:\.\d{2})?)', full_text, re.IGNORECASE)

            if income_match:
                extracted_data['monthly_income'] = float(income_match.group(1).replace(',', ''))
            if bank_match:
                extracted_data['bank_assets'] = float(bank_match.group(1).replace(',', ''))

    except Exception as e:
        print(f"Error reading PDF: {e}")

    return extracted_data


# ==========================================
# STEP 2: CREATE SAMPLE PDF (FOR TESTING)
# ==========================================
# Testing ke liye hum Colab par ek dummy Salary Slip PDF bana rahe hain
from fpdf import FPDF

pdf = FPDF()
pdf.add_page()
pdf.set_font("Arial", size=12)
pdf.cell(200, 10, txt="EMPLOYEE SALARY SLIP", ln=1, align='C')
pdf.cell(200, 10, txt="-----------------------------------", ln=2, align='C')
pdf.cell(200, 10, txt="Employee Name: Anmol Gola", ln=3)
pdf.cell(200, 10, txt="Net Pay: 100000", ln=4)           # Monthly Income = ₹1,00,000
pdf.cell(200, 10, txt="Bank Balance: 500000", ln=5)        # Bank Balance = ₹5,00,000
pdf.output("sample_salary_slip.pdf")

print("Sample PDF 'sample_salary_slip.pdf' generated!")


# ==========================================
# STEP 3: END-TO-END AUTOMATED INFERENCE
# ==========================================
def process_pdf_and_predict_loan(pdf_file_path, cibil_score=750, loan_term_years=10, dependents=2):
    # Step A: Parse PDF
    pdf_info = extract_financials_from_pdf(pdf_file_path)
    monthly_inc = pdf_info['monthly_income'] if pdf_info['monthly_income'] > 0 else 50000.0
    bank_bal = pdf_info['bank_assets'] if pdf_info['bank_assets'] > 0 else 200000.0

    annual_inc = monthly_inc * 12.0

    # Step B: Pass extracted features to XGBoost model
    input_payload = pd.DataFrame([{
        'no_of_dependents': dependents,
        'income_annum': annual_inc,
        'loan_term': loan_term_years,
        'cibil_score': cibil_score,
        'residential_assets_value': bank_bal * 2,
        'commercial_assets_value': 0,
        'luxury_assets_value': 0,
        'bank_asset_value': bank_bal,
        'monthly_income': monthly_inc
    }])

    max_loan = model.predict(input_payload)[0]

    print("\n--- OCR & PREDICTION RESULT ---")
    print(f"Extracted Monthly Income: ₹{monthly_inc:,.2f}")
    print(f"Extracted Bank Balance: ₹{bank_bal:,.2f}")
    print(f"Predicted Max Loan Amount: ₹{max_loan:,.2f}")

# Run end-to-end test on the sample PDF
process_pdf_and_predict_loan("sample_salary_slip.pdf", cibil_score=780, loan_term_years=15)

ModuleNotFoundError: No module named 'fpdf'